**IAM (Identity and Access Management)** is the AWS service that controls *who* can do *what* in your AWS account.

Every API call made to AWS — whether from the console, CLI, or SDK — is authenticated and authorised through IAM. Getting IAM right is the foundation of a secure AWS architecture.

## The Root Account

When you create an AWS account, you start with a **root user** — an identity with unrestricted access to everything in the account.

| | Root User | IAM User |
|---|---|---|
| **Created by** | Signing up for AWS | You, via IAM |
| **Permissions** | Unlimited — cannot be restricted | Controlled by policies |
| **Can be deleted** | No | Yes |
| **Recommended for daily use** | Never | Yes |

### Root-only tasks
A few actions can *only* be done by root:
- Change account name, email, or root password
- Close the AWS account
- Restore IAM user permissions if all admin access is accidentally removed
- Subscribe to certain support plans

> **Best practice:** Lock away the root account. Enable MFA on it, then never use it again for day-to-day work.

## IAM Users

An **IAM user** represents a person or application that interacts with AWS.

Each user has:
- A unique name within the account
- Optional **console password** (for humans)
- Optional **access key ID + secret** (for programmatic access via CLI/SDK)
- Permissions granted via policies

### One identity per person
Never share IAM credentials. If two developers share one IAM user, you lose auditability — CloudTrail can't tell them apart. Create one IAM user per human.

### IAM users are global
IAM is a **global service** — users are not tied to any specific region.

## IAM Groups

A **group** is a collection of IAM users. Attach a policy to the group, and every user in that group inherits those permissions.

```
Group: Developers
  Policy: AmazonEC2FullAccess
  Policy: AmazonS3ReadOnlyAccess
  ├── User: alice
  ├── User: bob
  └── User: charlie
```

### Rules
- A user can belong to **multiple groups**
- Groups **cannot contain other groups** (no nesting)
- A user doesn't have to belong to any group (though this is unusual)

> **Best practice:** Assign permissions to groups, not individual users. When an employee joins or leaves, add/remove them from the group — no policy changes needed.

## IAM Policies

A **policy** is a JSON document that defines what actions are allowed or denied on which AWS resources.

### Policy structure

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject",
        "s3:PutObject"
      ],
      "Resource": "arn:aws:s3:::my-bucket/*"
    }
  ]
}
```

| Field | Purpose |
|---|---|
| `Version` | Policy language version — always use `"2012-10-17"` |
| `Effect` | `Allow` or `Deny` |
| `Action` | The AWS API operation(s) — e.g. `s3:GetObject`, `ec2:*` |
| `Resource` | The ARN(s) the action applies to — `*` means all resources |
| `Condition` | (Optional) extra conditions — e.g. only from a specific IP |

### Explicit Deny always wins
AWS evaluates policies with this priority:
1. **Explicit Deny** — always overrides everything
2. **Explicit Allow** — grants access
3. **Implicit Deny** — the default; if nothing allows it, it's denied

### Policy types

| Type | Managed by | Reusable? |
|---|---|---|
| **AWS Managed** | AWS | Yes — pre-built, read-only |
| **Customer Managed** | You | Yes — your own reusable policies |
| **Inline** | You | No — embedded directly in one user/group/role |

## IAM Roles

A **role** is an IAM identity with permissions — but unlike a user, it has no permanent credentials. Instead, it issues **temporary security credentials** when assumed.

### When to use roles

| Scenario | How it works |
|---|---|
| EC2 needs to read from S3 | Attach a role to the EC2 instance — no access keys stored on disk |
| Lambda needs to write to DynamoDB | Lambda assumes a role automatically at runtime |
| Cross-account access | Role in Account B trusts Account A; users in A can assume it |
| Federated identity (SSO, Google) | External users assume a role after authenticating externally |

### Trust policy
Every role has two policies:
- **Trust policy** — defines *who* can assume the role (the principal)
- **Permissions policy** — defines *what* the role can do

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": { "Service": "ec2.amazonaws.com" },
    "Action": "sts:AssumeRole"
  }]
}
```

> **Key insight:** Roles are the preferred way to grant AWS services access to other AWS services. Never embed access keys inside EC2 instances or Lambda functions.

## Multi-Factor Authentication (MFA)

MFA adds a second layer of protection on top of a password. Even if a password is stolen, the attacker still can't sign in without the second factor.

### MFA device types AWS supports

| Type | Example |
|---|---|
| Virtual MFA | Google Authenticator, Authy (TOTP) |
| Hardware TOTP token | Gemalto physical device |
| FIDO security key | YubiKey (U2F/WebAuthn) |
| Hardware OTP | Key fob from a third party |

> **Best practice:** Enable MFA on the root account and on all IAM users with console access.

## Exploring IAM with boto3

In [ ]:
import boto3
import json

iam = boto3.client('iam')

In [ ]:
# List all IAM users in the account
response = iam.list_users()
for user in response['Users']:
    print(f"{user['UserName']:<30} created: {user['CreateDate'].date()}")

In [ ]:
# List all IAM groups
response = iam.list_groups()
for group in response['Groups']:
    print(group['GroupName'])

In [ ]:
# List IAM roles and their trust principals
response = iam.list_roles()
for role in response['Roles'][:10]:  # first 10
    trust = role['AssumeRolePolicyDocument']['Statement'][0]['Principal']
    print(f"{role['RoleName']:<45} trusted by: {trust}")

In [ ]:
# Simulate whether a policy action would be allowed
# Useful for debugging IAM permissions without making real API calls
response = iam.simulate_principal_policy(
    PolicySourceArn='arn:aws:iam::123456789012:user/alice',  # replace with real ARN
    ActionNames=['s3:GetObject', 's3:DeleteObject'],
    ResourceArns=['arn:aws:s3:::my-bucket/*']
)
for result in response['EvaluationResults']:
    print(f"{result['EvalActionName']:<25} → {result['EvalDecision']}")

## IAM Best Practices (SAA Exam Checklist)

| Practice | Why |
|---|---|
| Don't use root for daily work | One compromised credential = full account loss |
| Enable MFA on root and privileged users | Second layer stops credential theft |
| Grant least privilege | Only the permissions needed, nothing more |
| Use groups to assign permissions | Easier to manage at scale than per-user policies |
| Use roles for services, not access keys | Temporary credentials; no rotation needed |
| Rotate access keys regularly | Limits the window of exposure if a key leaks |
| Use IAM Access Analyzer | Detects unintended resource sharing with external entities |
| Set a password policy | Enforce complexity, expiry, and reuse restrictions |

## Summary

| Concept | Key Takeaway |
|---|---|
| Root account | Unrestricted; lock it away, enable MFA, never use daily |
| IAM User | One identity per person; password for console, access keys for CLI/SDK |
| IAM Group | Assign policies to groups, not individuals; users inherit group permissions |
| IAM Policy | JSON document: Effect + Action + Resource; Explicit Deny always wins |
| IAM Role | Temporary credentials; preferred for services and cross-account access |
| MFA | Second factor on top of password; mandatory for root and privileged users |
| Least privilege | Start with no permissions; grant only what's needed |